In [3]:
pip install uproot awkward vector numpy matplotlib fsspec-xrootd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 70.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [fsspec-xrootd]


In [5]:
import uproot
import awkward as ak
import vector
import numpy as np
import matplotlib.pyplot as plt

vector.register_awkward()

# 1. Configuração e Leitura dos Dados
file_path = "root://eospublic.cern.ch//eos/opendata/cms/Run2016H/DoubleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/2510000/127C2975-1B1C-A046-AABF-62B77E757A86.root"
print(f"Abrindo arquivo: {file_path}")
tree = uproot.open(file_path)["Events"]

# 2. Extração das Variáveis dos Múons
muons = tree.arrays(
    ["Muon_pt", "Muon_eta", "Muon_phi", "Muon_mass", "Muon_charge"],
    library="ak"
)

# 3. Seleção de Múons de Qualidade
good_muons = muons[
    (muons.Muon_pt > 10) &
    (abs(muons.Muon_eta) < 2.4)
]

# 4. Construção de Pares e Massa Invariante
muon_vecs = ak.zip(
    {
        "pt": good_muons.Muon_pt,
        "eta": good_muons.Muon_eta,
        "phi": good_muons.Muon_phi,
        "mass": good_muons.Muon_mass,
        "charge": good_muons.Muon_charge,
    },
    with_name="PtEtaPhiMCandidate",
)

pairs = ak.combinations(muon_vecs, 2, fields=["mu1", "mu2"])
opposite_charge_pairs = pairs[pairs.mu1.charge != pairs.mu2.charge]

invariant_mass = (opposite_charge_pairs.mu1 + opposite_charge_pairs.mu2).mass

mass_values = ak.flatten(invariant_mass, axis=None)
mass_values = ak.to_numpy(mass_values)

# 5. Visualização (com a raw string corrigida)
plt.figure(figsize=(10, 6))
plt.hist(
    mass_values,
    bins=100,
    range=(40, 140),
    histtype="step",
    color="black",
    label="CMS Open Data (Run 2016)"
)

plt.title("Reconstrução do Bóson Z → μμ")
plt.xlabel(r"Massa invariante $m_{\mu\mu}$ [GeV]")  # <--- Correção aqui
plt.ylabel("Número de eventos / (1 GeV)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig("zmumu_mass.png", dpi=150)
print("Histograma salvo como 'zmumu_mass.png'")
plt.show()

Abrindo arquivo: root://eospublic.cern.ch//eos/opendata/cms/Run2016H/DoubleMuon/NANOAOD/UL2016_MiniAODv2_NanoAODv9-v1/2510000/127C2975-1B1C-A046-AABF-62B77E757A86.root


TypeError: no numpy.add overloads for custom types: PtEtaPhiMCandidate, PtEtaPhiMCandidate